In [26]:
import pandas as pd

In [27]:
df = pd.read_csv('data/marketplace_products.csv')
df.head()


,platform,category,search_keyword,product_id,product_name,brand,price,original_price,discount_percent,currency,rating,review_count,sold_count,seller_name,seller_location,seller_rating,availability,product_url,crawl_time_utc
0,tiki,Làm đẹp,kem chống nắng,630777,Kem chống nắng Sunplay cực mạnh dạng sữa Sunpl...,Sunplay,167000.0,NaN,NaN,VND,4.8,425.0,NaN,NaN,NaN,NaN,InStock,https://tiki.vn/kem-chong-nang-sunplay-cuc-man...,2026-10-18T09:16:41+00:00
1,tiki,Làm đẹp,kem chống nắng,1313873,Kem chống nắng Sunplay mát lạnh giải nhiệt dạn...,Sunplay,86000.0,NaN,NaN,VND,4.8,221.0,NaN,NaN,NaN,NaN,InStock,https://tiki.vn/kem-chong-nang-sunplay-mat-lan...,2026-10-18T09:16:41+00:00
2,tiki,Làm đẹp,kem chống nắng,45915523,Kem Chống Nắng Clarins UV Plus Multi Protectio...,Clarins,1090000.0,NaN,NaN,VND,NaN,NaN,NaN,NaN,NaN,NaN,InStock,https://tiki.vn/kem-chong-nang-clarins-uv-plus...,2026-10-18T09:16:41+00:00
3,tiki,Làm đẹp,kem chống nắng,251536647,Kem chống nắng TENAMYD SPF50/PA+++,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://tiki.vn/kem-chong-nang-tenamyd-spf50-p...,2026-10-18T09:16:41+00:00
4,tiki,Làm đẹp,kem chống nắng,248355431,Kem Chống Nắng Kiềm Dầu Farmona Sun Face Cream...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://tiki.vn/kem-chong-nang-kiem-dau-farmon...,2026-10-18T09:16:41+00:00


In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1050 entries, 0 to 1049
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   platform          1050 non-null   object 
 1   category          1050 non-null   object 
 2   search_keyword    1050 non-null   object 
 3   product_id        1050 non-null   object 
 4   product_name      1050 non-null   object 
 5   brand             12 non-null     object 
 6   price             1026 non-null   float64
 7   original_price    318 non-null    float64
 8   discount_percent  768 non-null    float64
 9   currency          1026 non-null   object 
 10  rating            300 non-null    float64
 11  review_count      325 non-null    float64
 12  sold_count        832 non-null    object 
 13  seller_name       0 non-null      float64
 14  seller_location   568 non-null    object 
 15  seller_rating     0 non-null      float64
 16  availability      48 non-null     object 


In [29]:
df.drop(columns=['brand', 'search_keyword', 'seller_name', 'seller_rating', 'availability', 'product_url', 'crawl_time_utc'], inplace = True)

## EDA bổ sung
Các cell dưới đây chỉ thống kê và gắn cờ vấn đề; không thay đổi dữ liệu.

In [30]:
# Tóm tắt cấu trúc và số lượng theo sàn
print(f"Số dòng: {len(df):,} | Số cột: {df.shape[1]}")
display(df.groupby("platform").size().rename("product_count").to_frame())
display(pd.crosstab(df["platform"], df["category"]))

Số dòng: 1,050 | Số cột: 12


,product_count
platform,
lazada,350
shopee,350
tiki,350


category,Làm đẹp,Nhà cửa & đời sống,Sức khỏe,Thời trang nữ,Thời trang trẻ em,Thực phẩm & đồ uống,Văn phòng phẩm,health,Điện gia dụng,Điện tử
platform,,,,,,,,,,
lazada,39,39,39,37,39,39,41,0,38,39
shopee,35,35,35,35,35,35,35,35,35,35
tiki,52,48,25,50,25,40,33,0,29,48


In [31]:
# Tỷ lệ null toàn bộ và theo sàn
missing = pd.DataFrame({
    "null_count": df.isna().sum(),
    "null_pct": (df.isna().mean() * 100).round(1),
}).sort_values("null_pct", ascending=False)
display(missing)
missing_by_platform = pd.concat(
    {platform: group.isna().mean().mul(100).round(1) for platform, group in df.groupby("platform")},
    axis=1,
).T
display(missing_by_platform)

,null_count,null_pct
rating,750,71.4
original_price,732,69.7
review_count,725,69.0
seller_location,482,45.9
discount_percent,282,26.9
sold_count,218,20.8
price,24,2.3
currency,24,2.3
platform,0,0.0
category,0,0.0


,platform,category,product_id,product_name,price,original_price,discount_percent,currency,rating,review_count,sold_count,seller_location
lazada,0.0,0.0,0.0,0.0,0.0,100.0,16.3,0.0,100.0,9.7,9.7,25.4
shopee,0.0,0.0,0.0,0.0,0.0,9.1,1.1,0.0,16.9,100.0,16.3,12.3
tiki,0.0,0.0,0.0,0.0,6.9,100.0,63.1,6.9,97.4,97.4,36.3,100.0


In [32]:
# Kiểm tra duplicate theo dòng, theo khóa sản phẩm và theo URL
print("Duplicate toàn dòng:", df.duplicated().sum())
print("Duplicate (platform, product_id):", df.duplicated(["platform", "product_id"]).sum())
if "product_url" in df.columns:
    print("Duplicate (platform, product_url):", df.duplicated(["platform", "product_url"]).sum())

Duplicate toàn dòng: 0
Duplicate (platform, product_id): 0


In [33]:
# Phân phối các chỉ số dạng số; bản gốc không bị ghi đè
numeric_cols = [c for c in ["price", "original_price", "discount_percent", "rating", "review_count"] if c in df.columns]
numeric = df[numeric_cols].apply(pd.to_numeric, errors="coerce")
display(numeric.describe(percentiles=[.01, .25, .5, .75, .95, .99]).T)
if "price" in numeric:
    print("Giá bằng 0:", numeric["price"].eq(0).sum())
if "rating" in numeric:
    invalid_rating = numeric["rating"].notna() & ~numeric["rating"].between(0, 5)
    print("Rating ngoài [0, 5]:", invalid_rating.sum())
    display(df.loc[invalid_rating, ["platform", "product_id", "product_name", "rating"]].head(20))

,count,mean,std,min,1%,25%,50%,75%,95%,99%,max
price,1026.0,425630.888889,854797.507780,0.0,6568.250,70125.0,163500.0,388750.0,1677000.0,5817000.000,6990000.0
original_price,318.0,553737.034591,915251.329783,20000.0,32840.000,134000.0,265750.0,553575.0,2200000.0,4557700.000,8064000.0
discount_percent,768.0,35.493490,19.792666,1.0,2.000,19.0,35.0,49.0,70.0,89.000,100.0
rating,300.0,4.851667,0.340023,1.0,3.497,4.8,4.9,5.0,5.0,5.301,5.4
review_count,325.0,609.498462,1597.768435,0.0,0.000,9.0,46.0,366.0,3128.6,7605.000,13962.0


Giá bằng 0: 4
Rating ngoài [0, 5]: 9


,platform,product_id,product_name,rating
577,shopee,768354594_17570662369,Tai nghe Lenovo LP40 ProTWS Tai nghe Bluetooth...,5.3
580,shopee,1585829434_43979328729,Tai Nghe Ngủ Bluetooth 5.4 Không Dây Mini Ôm T...,5.4
587,shopee,131195741_6938221363,Ad,5.3
590,shopee,1356437574_51450584643,"Tai nghe Bluetooth 5.3 GOOJODOQ, âm thanh nổi ...",5.3
593,shopee,1211891129_49019305430,Tai Nghe Nhét Tai Không Dây Bluetooth 5.4 Kiip...,5.4
597,shopee,808014822_25221952029,Ad,5.3
605,shopee,1827254437_40531590936,Tai Nghe Móc Tai Q16S Bluetooth 5.4 Thể Thao C...,5.4
606,shopee,18113796_56968039637,LamKa Shop: Tai Nghe Sony WF-1000XM6 Cao Cấp C...,5.3
607,shopee,1827273422_57915415722,Tai Nghe Bluetooth Thể Thao Philips TAA1708 Ch...,5.3


In [34]:
# Gắn cờ ngoại lệ giá theo từng sàn và danh mục; không xóa hay winsorize
if "price" in df.columns:
    price_check = df[["platform", "category", "price"]].copy()
    price_check["category"] = price_check["category"].replace({"health": "Sức khỏe"})
    price_check["price"] = pd.to_numeric(price_check["price"], errors="coerce").replace(0, pd.NA)
    summaries = []
    for (platform, category), group in price_check.groupby(["platform", "category"], dropna=False):
        x = group["price"].dropna()
        if x.empty:
            continue
        q1, q3 = x.quantile([.25, .75])
        iqr = q3 - q1
        low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        summaries.append({"platform": platform, "category": category, "n": x.size, "q1": q1, "median": x.median(), "q3": q3, "lower_fence": low, "upper_fence": high, "outside_iqr_fences": ((x < low) | (x > high)).sum()})
    display(pd.DataFrame(summaries).set_index(["platform", "category"]))

n         q1     median         q3  \
platform category                                                   
lazada   Làm đẹp              39    83850.0   159000.0   307000.0   
         Nhà cửa & đời sống   39    60875.5   136000.0   195000.0   
         Sức khỏe             39   129670.0   310000.0   442500.0   
         Thời trang nữ        37    44000.0    49999.0    69500.0   
         Thời trang trẻ em    39    39000.0    60000.0    81000.0   
         Thực phẩm & đồ uống  39   119500.0   155000.0   208925.0   
         Văn phòng phẩm       41    18000.0    58000.0    96900.0   
         Điện gia dụng        38   635500.0   810000.0  1007250.0   
         Điện tử              39    59200.0    99000.0   309000.0   
shopee   Làm đẹp              35   161500.0   253300.0   330625.0   
         Nhà cửa & đời sống   35   127067.5   215438.0   312924.5   
         Sức khỏe             70    77500.0   148063.5   233075.0   
         Thời trang nữ        35    62000.0   105000.0   150308.0   
         Thời trang trẻ em    35    59125.0    71600.0    99500.0   
         Thực phẩm & đồ uống  35    52550.0    89000.0   214497.5   
         Văn phòng phẩm       31    14328.5    31500.0    84500.0   
         Điện gia dụng        35   747120.0  1459200.0  1861740.5   
         Điện tử              35   184365.0   234999.0   548564.0   
tiki     Làm đẹp              50   182500.0   314500.0   504500.0   
         Nhà cửa & đời sống   42    98000.0   246500.0   562500.0   
         Sức khỏe             25   125000.0   449000.0   849000.0   
         Thời trang nữ        50    79725.0   112000.0   185000.0   
         Thời trang trẻ em    25    99000.0   198000.0   229000.0   
         Thực phẩm & đồ uống  40    78000.0   175000.0   247500.0   
         Văn phòng phẩm       24    44450.0    58750.0   642475.0   
         Điện gia dụng        23  1501500.0  2269000.0  2603500.0   
         Điện tử              47   402000.0   699000.0  2716000.0   

                              lower_fence  upper_fence  outside_iqr_fences  
platform category                                                           
lazada   Làm đẹp               -250875.00    641725.00                   4  
         Nhà cửa & đời sống    -140311.25    396186.75                   4  
         Sức khỏe              -339575.00    911745.00                   1  
         Thời trang nữ            5750.00    107750.00                   3  
         Thời trang trẻ em      -24000.00    144000.00                   1  
         Thực phẩm & đồ uống    -14637.50    343062.50                   5  
         Văn phòng phẩm        -100350.00    215250.00                   2  
         Điện gia dụng           77875.00   1564875.00                   3  
         Điện tử               -315500.00    683700.00                   5  
shopee   Làm đẹp                -92187.50    584312.50                   1  
         Nhà cửa & đời sống    -151718.00    591710.00                   4  
         Sức khỏe              -155862.50    466437.50                   5  
         Thời trang nữ          -70462.00    282770.00                   2  
         Thời trang trẻ em       -1437.50    160062.50                   1  
         Thực phẩm & đồ uống   -190371.25    457418.75                   2  
         Văn phòng phẩm         -90928.75    189757.25                   4  
         Điện gia dụng         -924810.75   3533671.25                   2  
         Điện tử               -361933.50   1094862.50                   2  
tiki     Làm đẹp               -300500.00    987500.00                   4  
         Nhà cửa & đời sống    -598750.00   1259250.00                   2  
         Sức khỏe              -961000.00   1935000.00                   0  
         Thời trang nữ          -78187.50    342912.50                   2  
         Thời trang trẻ em      -96000.00    424000.00                   0  
         Thực phẩm & đồ uống   -176250.00    501750.00                   2  
         Văn phòng phẩm    

## Tiền xử lý đã thống nhất
Mỗi cell dưới đây làm một bước và tạo `df_clean` từ `df`. Các giá trị null được giữ nguyên, không điền median. CSV nguồn không bị ghi đè. Chạy lần lượt các cell theo thứ tự.

In [35]:
# 1. Tạo bản sao làm sạch và chuẩn hóa nhãn danh mục
df_clean = df.copy()
df_clean["category"] = df_clean["category"].astype("string").str.strip()
df_clean["category"] = df_clean["category"].replace({"health": "Sức khỏe"})
print("Số danh mục health đã chuẩn hóa:", int(df["category"].astype("string").str.strip().eq("health").sum()))

Số danh mục health đã chuẩn hóa: 35


In [36]:
# 2. Chuẩn hóa kiểu số và xử lý các giá trị không hợp lệ
import numpy as np
for col in ["price", "original_price", "discount_percent", "rating", "review_count"]:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# Giá 0 không hợp lệ trong dữ liệu sản phẩm này; chuyển thành null.
zero_price = df_clean["price"].eq(0)
df_clean.loc[zero_price, "price"] = np.nan

# Rating ngoài [0, 5] được xem là lỗi parse và chuyển thành null.
invalid_rating = df_clean["rating"].notna() & ~df_clean["rating"].between(0, 5)
df_clean.loc[invalid_rating, "rating"] = np.nan

# Toàn bộ dữ liệu thuộc các sàn Việt Nam; điền VND nếu currency trống.
currency = df_clean["currency"].astype("string").str.strip()
currency = currency.mask(currency.eq(""), pd.NA)
df_clean["currency"] = currency.fillna("VND")
print("Giá 0 chuyển null:", int(zero_price.sum()))
print("Rating ngoài [0, 5] chuyển null:", int(invalid_rating.sum()))

Giá 0 chuyển null: 4
Rating ngoài [0, 5] chuyển null: 9


In [37]:
# 3. Giữ lượt bán gốc và tạo ước lượng số lượt bán
import re
def parse_sold_count(value):
    if pd.isna(value):
        return np.nan, pd.NA

    text = str(value).strip()
    match = re.fullmatch(r"(\d+(?:[.,]\d+)?)\s*([kKmM])?\s*(\+)?", text)
    if not match:
        return np.nan, pd.NA

    amount = float(match.group(1).replace(",", "."))
    suffix = (match.group(2) or "").lower()
    multiplier = {"": 1, "k": 1_000, "m": 1_000_000}[suffix]
    estimate = round(amount * multiplier)
    is_lower_bound = bool(match.group(3))
    return estimate, is_lower_bound

parsed_sold = df_clean["sold_count"].apply(parse_sold_count)
df_clean["sold_count_estimate"] = parsed_sold.map(lambda result: result[0])
df_clean["sold_count_is_lower_bound"] = pd.array(
    parsed_sold.map(lambda result: result[1]), dtype="boolean"
)
print("Lượt bán quy đổi được:", int(df_clean["sold_count_estimate"].notna().sum()))
print("Ước lượng là cận dưới:", int(df_clean["sold_count_is_lower_bound"].sum(skipna=True)))

Lượt bán quy đổi được: 832
Ước lượng là cận dưới: 100


In [38]:
# 4. Gắn cờ ngoại lệ giá theo từng sàn và danh mục
df_clean["price_outlier"] = pd.Series(
    pd.NA, index=df_clean.index, dtype="boolean"
)

for _, group in df_clean.groupby(["platform", "category"], dropna=False):
    prices = group["price"].dropna()
    if prices.empty:
        continue

    q1, q3 = prices.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    flags = prices.lt(lower_fence) | prices.gt(upper_fence)
    df_clean.loc[prices.index, "price_outlier"] = flags.to_numpy()

print("Giá được gắn cờ ngoại lệ:", int(df_clean["price_outlier"].sum()))

Giá được gắn cờ ngoại lệ: 73


In [39]:
# 5. Xuất file mới và xem nhanh kết quả
output_path = "data/marketplace_products_clean.csv"
df_clean.to_csv(output_path, index=False, encoding="utf-8-sig")

display(df_clean.groupby("platform").size().rename("product_count").to_frame())
display(df_clean[["price", "original_price", "discount_percent", "rating", "review_count", "sold_count_estimate"]].describe().T)
print("Số dòng:", len(df_clean))
print("Số duplicate theo khóa:", df_clean.duplicated(["platform", "product_id"]).sum())
print("Đã lưu:", output_path)

,product_count
platform,
lazada,350
shopee,350
tiki,350


,count,mean,std,min,25%,50%,75%,max
price,1022.0,427296.763209,856054.238893,51.0,71091.25,165000.0,390000.0,6990000.0
original_price,318.0,553737.034591,915251.329783,20000.0,134000.00,265750.0,553575.0,8064000.0
discount_percent,768.0,35.493490,19.792666,1.0,19.00,35.0,49.0,100.0
rating,291.0,4.836770,0.334234,1.0,4.80,4.9,5.0,5.0
review_count,325.0,609.498462,1597.768435,0.0,9.00,46.0,366.0,13962.0
sold_count_estimate,832.0,4373.328125,23096.499019,1.0,11.00,118.5,1000.0,400000.0


Số dòng: 1050
Số duplicate theo khóa: 0
Đã lưu: data/marketplace_products_clean.csv
